<div style="background:#102A43;color:#ffffff;border-radius:11px;padding:30px 28px 24px;border-top:7px solid #18B7AC;">
<div style="color:#67DED3;letter-spacing:2px;font-weight:700;font-size:12px;">INTRODUCTION TO MACHINE LEARNING · GAZA SKY GEEKS</div>
<div style="font-size:32px;font-weight:750;line-height:1.25;margin:12px 0 9px;">Lecture 05 · Descriptive Statistics<br>for Data Science</div>
<div style="font-size:15px;color:#DDEAF3;">Instructor: Diaeddin Rimawi &nbsp; | &nbsp; 31 October 2026</div>
<div style="font-size:17px;color:#ffffff;border-top:1px solid #466178;padding-top:16px;margin-top:18px;"><strong>Today's challenge:</strong> We can see patterns in our data. How can we describe and compare them precisely using numbers?</div>
</div>

**Our numerical investigation**

In **L03**, we used Pandas to inspect and summarize measurements. In **L04**, we explored histograms, box plots, and scatter plots to see distributions and relationships. Today, we will explain those same observations with **meaningful numerical measures**. We will continue with **Palmer Penguins**, without changing the dataset.


### What you will be able to do

By the end of this lecture, you should be able to:

- Choose and interpret **mean, median, and mode** to describe a typical observation.
- Explain and calculate **range, variance, and standard deviation** to describe variation.
- Read and calculate **quartiles, percentiles, IQR, and a five-number summary**.
- Explain the **1.5 × IQR rule**, including what box-plot whiskers actually represent.
- Interpret **Pearson correlation** in the full dataset and within penguin species.
- Write a short, evidence-based statistical description of a group, while recognizing what the summaries *cannot* tell us.

This lecture describes our observed data. It does **not** introduce hypothesis testing, statistical prediction, or data cleaning.

### A familiar dataset, one new set of questions

We continue with the **Palmer Penguins dataset** from L03 and L04. It records measurements of **344 penguins** across **three species**. Each row is an observation; each column is an attribute.

| Attribute | Meaning |
|:--|:--|
| `species`, `island`, `sex` | Categorical attributes |
| `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm` | Numerical measurements, in mm |
| `body_mass_g` | Numerical measurement, in grams |
| `year` | Year of observation (treated as a recorded year, not as a measurement to average) |


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
NAVY = "#102A43"
TEAL = "#158B94"
SPECIES_ORDER = ["Adelie", "Chinstrap", "Gentoo"]
SPECIES_COLORS = {"Adelie": "#158B94", "Chinstrap": "#DB9137", "Gentoo": "#557AB5"}

path = Path("data/penguins.csv")
# assert path.is_file(), "Open this notebook from the L05 folder (with data/penguins.csv)."
penguins = pd.read_csv(path)
# assert penguins.shape == (344, 8), "Unexpected penguins dataset: verify the bundled CSV."
display(penguins.head())

In [ ]:
print(f"All penguin observations: {len(penguins)}")
print(f"Recorded body masses: {penguins['body_mass_g'].count()}")
print(f"Missing body masses: {penguins['body_mass_g'].isna().sum()}")
print(f"Recorded flipper-length/body-mass pairs: {penguins[['flipper_length_mm', 'body_mass_g']].dropna().shape[0]}")
display(penguins['species'].value_counts().rename("Observations").to_frame())

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
    <div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">01 · VISUAL TO NUMERICAL</div>
    <div style="font-size:23px;font-weight:700;line-height:1.4;">From visual patterns to numerical descriptions</div>
    <div style="font-size:14px;color:#DDEAF3;margin-top:4px;">A histogram tells a story. Which numbers would make that story precise?</div>
    </div>

Recall the **body-mass histogram from L04**: each bar counts how many penguins belong in a body-mass interval. It helps us see the **shape and concentration** of the observations, but it does not directly state an exact typical mass or quantify their variability.

Before running the cell below, try to answer:

- Near which masses are many observations concentrated?
- Does the histogram show one tight group, or could it contain different groups?
- Can you read the exact average and standard deviation from the graph alone?

In L04, we also saw that Adelie and Chinstrap have similar mean body masses, but a mean alone cannot tell us whether their distributions are alike. We will return to that comparison with measures of spread.

**Descriptive statistics** are numerical measures that summarize the data we actually observed. They complement plots; they do not replace them. Claims about all penguins outside this dataset would need more information about how observations were collected.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.hist(penguins["body_mass_g"].dropna(), bins=17, color="#158B94", edgecolor="white")
ax.set(title="Penguin body mass: our starting visual observation",
       xlabel="Body mass (g)", ylabel="Number of penguins")
plt.show()

We can now ask three useful numerical questions:

1. **Center:** Where is a typical value?
2. **Spread:** How different are the observations from one another?
3. **Relationship:** Do two measurements tend to change together?


<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
    <div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">02 · TYPICAL VALUES</div>
    <div style="font-size:23px;font-weight:700;line-height:1.4;">Measures of central tendency</div>
    <div style="font-size:14px;color:#DDEAF3;margin-top:4px;">What is a typical body mass for a penguin?</div>
    </div>

### The mean: sharing the total equally

Suppose five penguins have masses measured in simplified units: **3, 4, 4, 5, 9**. Their total is 25. Dividing the total equally among five observations gives **5**.

The **arithmetic mean** (usually called the average) is:

$$
\bar{x}=\frac{1}{n}\sum_{i=1}^{n} x_i
$$

Here, $x_i$ is observation $i$, $n$ is the number of valid observations, $\sum$ means *add them together*, and $\bar{x}$ is the mean.

$$
\bar{x}=\frac{3+4+4+5+9}{5}=5.
$$

The mean uses **every** observation. That makes it useful, but also means that unusually large or small values can pull it away from what most observations look like.

In [ ]:
toy_center = pd.Series([3, 4, 4, 5, 9], name="Illustrative masses")
print("Manual mean:", toy_center.sum() / toy_center.count())
print("Pandas mean:", toy_center.mean())
print("NumPy mean:", np.mean(toy_center))

### The median: the middle position

Sort the values: **3, 4, 4, 5, 9**. The middle (third) value is **4**. This is the **median**.

- If $n$ is **odd**, the median is the middle value after sorting.
- If $n$ is **even**, it is the average of the two middle values. For example, the median of **3, 4, 7, 10** is $(4+7)/2=5.5$.

The median is about the **position** of the observations, not their total. It is therefore much less affected by the size of an extreme value.

### The mode: the most common value

The **mode** is a value that occurs most often. Here, **4** appears twice, so the mode is **4**. Some datasets have several modes or no uniquely most frequent value. The mode is also useful for **categorical attributes**, for example the most frequently observed penguin species.

Mean and median are used with numerical measurements. A *mode* can be used with both numerical and categorical observations.

In [ ]:
print(f"Mean: {toy_center.mean():.2f}")
print(f"Median: {toy_center.median():.2f}")
print(f"Mode(s): {toy_center.mode().tolist()}")
print("Most common species:", penguins["species"].mode().tolist())

### Why the choice of center matters

Start with the five values **3, 4, 4, 5, 9**. Now imagine that the last observation had been **29** instead of 9. Only one value changes. What happens to the mean and the median?

The graph below shows this example, not the penguin dataset.

In [ ]:
ordinary = pd.Series([3, 4, 4, 5, 9])
with_extreme = pd.Series([3, 4, 4, 5, 29])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, values, label in zip(axes, [ordinary, with_extreme],
                             ["Original example", "One unusually large value"]):
    ax.scatter(values, np.ones(len(values)), s=95, color="#58A9B5", zorder=3, label="Observations")
    ax.axvline(values.mean(), color="#D8753C", linewidth=2, label=f"Mean = {values.mean():.1f}")
    ax.axvline(values.median(), color=NAVY, linestyle="--", linewidth=2,
               label=f"Median = {values.median():.1f}")
    ax.set(xlabel="Illustrative value", title=label, xlim=(0, 32), ylim=(0.75, 1.25))
    ax.set_yticks([])
    ax.legend(loc="upper right", fontsize=8)
fig.suptitle("Changing one observation can move the mean much more than the median")
plt.tight_layout()
plt.show()

**Interpretation:** With an extreme value, the mean increases from 5 to 9, while the median stays at 4. If we want to describe the value near the middle of the ordered observations, the median can be more representative. The mean is still a valid summary of the arithmetic total; we simply need to know what question it answers.

Neither measure tells us how spread out the values are. That will motivate our next section.

<div style="border-left:5px solid #18B7AC;background:#EEF8F7;padding:12px 16px;border-radius:5px;"><strong style="color:#102A43;">ACTIVITY 1 — Which average tells the better story?</strong></div>

Consider these **illustrative weekly waiting times**, in minutes: `4, 5, 5, 6, 30`.

**Your task:** Calculate the mean and median. Which value better describes the experience of a *typical* week?

*Try the question before opening the suggested answer. The next code cell is yours to edit.*

In [ ]:
# Activity 1: Try your calculation here.
waiting_times = pd.Series([4, 5, 5, 6, 30])
# Use .mean() and .median(); then explain your choice in words.
print("Mean:", waiting_times.mean())
print("Median:", waiting_times.median())

<details><summary><strong>Suggested answer — click to expand after trying</strong></summary>

The mean is **10 minutes**, and the median is **5 minutes**. For a typical week, 5 minutes is a better description: four of the five weeks are between 4 and 6 minutes. But the 30-minute week is real in this example and should not be hidden. A useful report might say: **median 5 minutes, with one unusually long week of 30 minutes**. The mean still correctly reflects the total waiting time across all weeks.

</details>

### Returning to the real penguins

Our measurements have physical units. A mean body mass of about 4,200 is **4,200 g**.” Use the count alongside the statistic so readers know how many values contributed.

In [ ]:
body_mass = penguins["body_mass_g"]
center = pd.Series({
    "Valid masses (n)": int(body_mass.count()),
    "Mean (g)": body_mass.mean(),
    "Median (g)": body_mass.median(),
    "Mode (g)": body_mass.mode().iloc[0],
})
display(center.to_frame("Overall body mass").round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.hist(body_mass.dropna(), bins=17, color="#86BDC2", edgecolor="white")
ax.axvline(body_mass.mean(), color="#B85232", linewidth=2.5,
           label=f"Mean: {body_mass.mean():.0f} g")
ax.axvline(body_mass.median(), color=NAVY, linestyle="--", linewidth=2.5,
           label=f"Median: {body_mass.median():.0f} g")
ax.set(title="The mean and median add precision to the histogram",
       xlabel="Body mass (g)", ylabel="Number of penguins")
ax.legend()
plt.show()

The **overall** mean is about **4,202 g**, and the median is **4,050 g** (342 valid masses). But is an overall center an equally good summary for **all three species**? That question brings back the `groupby()` operation.

In [ ]:
group_center = (penguins.groupby("species" )["body_mass_g"]
                .agg(n="count", mean_g="mean", median_g="median")
                .reindex(SPECIES_ORDER))
display(group_center.round(1))

**Read the numbers:** Adelie and Chinstrap have similar mean body masses (about **3,701 g** and **3,733 g**), while Gentoo is heavier on average (about **5,076 g**). The overall mean mixes these groups and is not the typical value of every species. A separate summary can answer a more meaningful question.

**Remember our L04 question:** The Adelie and Chinstrap means were close, but the corresponding box plots did not look identical. Next we will put numbers on that difference.

**Bridge to spread:** Even two species with similar means can differ substantially in how much their body masses vary.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
    <div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">03 · HOW VALUES VARY</div>
    <div style="font-size:23px;font-weight:700;line-height:1.4;">Measures of variability</div>
    <div style="font-size:14px;color:#DDEAF3;margin-top:4px;">Two groups have the same mean. Are their observations similar?</div>
    </div>

### Equal means can hide different stories

Imagine these two small sets of *illustrative measurements*:

- Group A: **4, 5, 6, 7, 8**
- Group B: **0, 3, 6, 9, 12**

Both have mean **6**. However, Group A stays close to 6 while Group B reaches much farther away.

A center alone cannot describe **variability** (also called *spread*). To compare two groups, we need a second number.

In [ ]:
group_a = pd.Series([4, 5, 6, 7, 8])
group_b = pd.Series([0, 3, 6, 9, 12])

fig, ax = plt.subplots(figsize=(9, 3.3))
ax.scatter(group_a, [1] * len(group_a), s=110, color=TEAL, label="A: close together")
ax.scatter(group_b, [0] * len(group_b), s=110, color="#D68D3B", label="B: more spread out")
ax.axvline(6, color=NAVY, linestyle="--", label="Shared mean = 6")
ax.set(xlim=(-1, 13), ylim=(-0.6, 1.6), xlabel="Illustrative measurement",
       title="Same mean, different spread")
ax.set_yticks([0, 1], ["Group B", "Group A"])
ax.legend(loc="upper right", fontsize=9)
plt.show()

### Range: distance from the smallest to the largest

The **range** is easy to understand:

$$
\mathrm{Range}=x_{\max}-x_{\min}.
$$

- A: $8-4=4$
- B: $12-0=12$

The range captures the full width, but depends **only on two observations** (the minimum and maximum). A single unusual value may greatly change it. We need a way to consider how *all* values vary.

In [ ]:
for label, values in [("A", group_a), ("B", group_b)]:
    print(f"Group {label}: mean={values.mean():.1f}, range={values.max()-values.min():.1f}")

### From deviations to variance

A **deviation** tells us how far one observation is from the mean:

$$
\text{deviation}_i=x_i-\bar{x}.
$$

For the example **2, 4, 6, 8, 10**, the mean is 6. Deviations are **−4, −2, 0, 2, 4**. They add up to zero: negative and positive differences cancel.

To measure spread, we **square** the differences before averaging them. Squaring also gives larger deviations greater influence:

| Value $x_i$ | Deviation $x_i-6$ | Squared deviation $(x_i-6)^2$ |
|---:|---:|---:|
| 2 | −4 | 16 |
| 4 | −2 | 4 |
| 6 | 0 | 0 |
| 8 | 2 | 4 |
| 10 | 4 | 16 |
| **Total** | **0** | **40** |

We have **40 units squared** of total squared deviation. How we divide it depends on what we are describing.

In [ ]:
variance_example = pd.Series([2, 4, 6, 8, 10])
example_mean = variance_example.mean()
steps = pd.DataFrame({
    "value": variance_example,
    "deviation": variance_example - example_mean,
    "squared_deviation": (variance_example - example_mean) ** 2,
})
display(steps)
print("Sum of squared deviations:", steps["squared_deviation"].sum())

### Population variance and sample variance

For the **full population** of $N$ values we are describing, population variance is:

$$
\sigma^2 = \frac{1}{N}\sum_{i=1}^{N}(x_i-\mu)^2.
$$

$\mu$ is the population mean, $N$ is its size, and $\sigma^2$ is the population variance. For our **five values**, treating those five as the complete group gives **40/5 = 8**.

Often our observations are considered a **sample** from some larger population. A widely used sample variance divides by $n-1$:

$$
s^2 = \frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2.
$$

Here $n$ is the number of observed values, $\bar{x}$ is their mean, and $s^2$ is sample variance. The same example gives **40/4 = 10**. Dividing by $n-1$ is a standard correction used when estimating variability from a sample; we will discuss sampling more fully in L06.


In [ ]:
print("Population variance (divide by n):", variance_example.var(ddof=0))
print("Sample variance (divide by n-1):", variance_example.var(ddof=1))
print("Pandas .var() default:", variance_example.var())
print("NumPy np.var() default:", np.var(variance_example))

**Python convention matters:** `pd.Series.var()` and `pd.Series.std()` default to `ddof=1`, which means a divisor of $n-1$. NumPy's `np.var()` and `np.std()` default to `ddof=0`, a divisor of $n$. Always check the convention when comparing results.

We will use **sample** standard deviations for the penguin summaries, stating the choice explicitly. These are still *descriptions of the observed values* in this lecture.

### Standard deviation: bring the units back

Variance has **squared units**. If our measurements were in grams, variance would be in **grams squared** ($\mathrm{g}^2$), which is harder to interpret.

The **standard deviation** is the square root of the variance:

$$
s=\sqrt{s^2}=\sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2}.
$$

For our five-value example, the sample standard deviation is $\sqrt{10}\approx 3.16$ in the **original units**. It gives a useful measure of typical distance around the mean, though it is not literally the mean absolute distance.

Standard deviation is never negative. It is zero only when all recorded values are identical. Like the mean, it can be strongly affected by unusual observations.

In [ ]:
print(f"Sample standard deviation, by hand: {np.sqrt(40/4):.3f}")
print(f"Pandas sample standard deviation: {variance_example.std():.3f}")
print(f"NumPy sample standard deviation: {np.std(variance_example, ddof=1):.3f}")

<div style="border-left:5px solid #18B7AC;background:#EEF8F7;padding:12px 16px;border-radius:5px;"><strong style="color:#102A43;">ACTIVITY 2 — Same average, different spread</strong></div>

Look again at Group A (`4, 5, 6, 7, 8`) and Group B (`0, 3, 6, 9, 12`).

**Your task:** Predict which group has the larger **sample** standard deviation and why. Check with `.std()`. Would the range alone tell the same story here?

*Try the question before opening the suggested answer. The next code cell is yours to edit.*

In [ ]:
# Activity 2: Explain your prediction before calculating.
print("A - mean:", group_a.mean(), "sample SD:", group_a.std())
print("B - mean:", group_b.mean(), "sample SD:", group_b.std())
# Now explain what the comparison tells you.

<details><summary><strong>Suggested answer — click to expand after trying</strong></summary>

Both means are **6**. Group A has sample SD $\sqrt{2.5}\approx1.58$, and Group B has sample SD $\sqrt{22.5}\approx4.74$. B has greater variability because its observations are farther from the shared mean. The ranges (4 versus 12) point in the same direction **in this example**, but range uses only the endpoints, while SD uses every observation.

</details>

### Variability in real penguin body mass

Now the units matter: we report standard deviation in **grams**. If Adelie and Chinstrap have similar means, do they also have similar spreads?

In [ ]:
variation = (penguins.groupby("species")["body_mass_g"]
             .agg(n="count", mean_g="mean", minimum_g="min", maximum_g="max",
                  variance_g2="var", std_g="std")
             .reindex(SPECIES_ORDER))
variation["range_g"] = variation["maximum_g"] - variation["minimum_g"]
display(variation[["n", "mean_g", "range_g", "variance_g2", "std_g"]].round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
sns.boxplot(data=penguins, x="species", y="body_mass_g", hue="species",
            order=SPECIES_ORDER, hue_order=SPECIES_ORDER,
            palette=SPECIES_COLORS, legend=False, ax=ax)
ax.set(title="Body mass by species: a box plot helps compare spread",
       xlabel="Penguin species", ylabel="Body mass (g)")
plt.show()

Read the table and plot together: Chinstrap has a sample SD of about **384 g**, Adelie about **459 g**, and Gentoo about **504 g**. The larger SD for Gentoo means its masses are more spread around its **own mean** on this measure.

The box plot gives another view of spread. But its box represents the **middle half**, not the standard deviation. To understand it properly, we need quartiles.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
    <div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">04 · THE MIDDLE HALF</div>
    <div style="font-size:23px;font-weight:700;line-height:1.4;">Percentiles, quartiles, and the middle 50%</div>
    <div style="font-size:14px;color:#DDEAF3;margin-top:4px;">How can we describe the middle part of a distribution?</div>
    </div>

### Percentiles: positions in an ordered dataset

Imagine sorting all measured body masses from smallest to largest. A **percentile** marks a position in this ordered set.

- The **25th percentile** is the value at or below which roughly **25%** of observations lie.
- The **50th percentile** is the **median**.
- The **75th percentile** is the value at or below which roughly **75%** lie.

We call these special percentiles **quartiles**:

| Measure | Name | Interpretation |
|:--|:--|:--|
| $Q_1$ | First quartile | Roughly 25% of values are at or below it |
| $Q_2$ | Second quartile (median) | Roughly 50% of values are at or below it |
| $Q_3$ | Third quartile | Roughly 75% of values are at or below it |

Each quarter contains **approximately one fourth of the ordered observations**. Quartiles are *boundary values*, not equal-width numerical intervals. They work for many numerical distributions; the data do **not** need to follow a normal distribution.

Because datasets can be small and may contain repeated values, the proportions are approximate. Software can use different conventions to define an exact percentile between observed values.

### Work through nine simple values

Consider this **illustrative sorted dataset**:

$$
2,\quad 4,\quad 4,\quad 6,\quad 8,\quad 10,\quad 10,\quad 12,\quad 14.
$$

There are nine values. The middle value is $Q_2=8$.

Using the default linear-interpolation convention of Pandas (which gives positions 25%, 50%, 75% in this example):

- $Q_1=4$
- $Q_2=8$
- $Q_3=10$

We can now measure the width of the **middle half** of the ordered values:

$$
\mathrm{IQR}=Q_3-Q_1=10-4=6.
$$

The **interquartile range (IQR)** is measured in the original units and does not use the smallest or largest value. It is therefore less sensitive to extreme observations than the full range.

Different quartile definitions can give slightly different answers in small samples, so state the convention when it matters. Here and throughout this lecture, we use Pandas `.quantile()` with its default **linear** interpolation.

In [ ]:
ordered_example = pd.Series([2, 4, 4, 6, 8, 10, 10, 12, 14])
q1, q2, q3 = ordered_example.quantile([0.25, 0.50, 0.75])
print(f"Q1 (25th percentile): {q1:g}")
print(f"Q2 (median): {q2:g}")
print(f"Q3 (75th percentile): {q3:g}")
print(f"IQR (Q3 - Q1): {q3 - q1:g}")
print("90th percentile:", ordered_example.quantile(0.90))

The 90th percentile is another position marker, but it is **not** another quartile. A percentile says where a value lies in the ordered observations; it does not say that values occupy equal distances between percentiles.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.1))
ax.scatter(ordered_example, np.zeros(len(ordered_example)), s=95,
           color="#4B9BAA", edgecolor="white", linewidth=1.4, zorder=3,
           label="Observed values")
ax.axvspan(q1, q3, color="#C0EDE4", alpha=0.7, label="Middle 50% interval")
for value, label, color in [(q1, "Q1 = 4", TEAL), (q2, "Median = 8", NAVY),
                            (q3, "Q3 = 10", "#A36935")]:
    ax.axvline(value, linestyle="--", color=color, linewidth=1.7)
    ax.text(value, 0.17, label, ha="center", color=color, fontsize=10)
ax.set(title="Quartiles mark positions; the middle-half interval has width IQR = 6",
       xlabel="Illustrative value", xlim=(0, 16), ylim=(-0.3, 0.46))
ax.set_yticks([])
ax.legend(loc="upper right", fontsize=9)
plt.show()

### The five-number summary

We can describe a distribution with five ordered landmarks:

$$
\underbrace{\min}_{\text{smallest}},\quad
\underbrace{Q_1}_{25\%},\quad
\underbrace{Q_2}_{\text{median}},\quad
\underbrace{Q_3}_{75\%},\quad
\underbrace{\max}_{\text{largest}}.
$$

In the example, they are **2, 4, 8, 10, 14**. Notice that the **mean and standard deviation are not part of the five-number summary**.

What each number contributes:

- Minimum and maximum show the observed extremes.
- $Q_2$ describes the middle position.
- $Q_1$ and $Q_3$ show the middle half, with IQR $Q_3-Q_1$.

These five values provide the basis for a **box plot**, though the usual box-plot *whiskers* are not always the minimum and maximum.

In [ ]:
five_number = pd.Series({
    "Minimum": ordered_example.min(),
    "Q1": ordered_example.quantile(0.25),
    "Median (Q2)": ordered_example.median(),
    "Q3": ordered_example.quantile(0.75),
    "Maximum": ordered_example.max(),
})
display(five_number.to_frame("Illustrative value"))

### Connecting the numbers to the box plots

In a conventional box plot:

- The **left/bottom edge** of the box marks $Q_1$, the line inside marks the **median**, and the **right/top edge** marks $Q_3$.
- The **box length** along the numerical axis is the **IQR**.
- A longer box means more spread among the **middle 50%** of measurements, not necessarily greater standard deviation.
- The whiskers and any isolated points follow a separate convention, which we will explain next.

This is the missing numerical connection to the plots you already know.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2.8))
ax.boxplot(ordered_example, orientation="horizontal", widths=0.45, patch_artist=True,
           boxprops={"facecolor": "#B1E5DE", "edgecolor": NAVY},
           medianprops={"color": "#B85232", "linewidth": 2.5},
           whiskerprops={"color": NAVY}, capprops={"color": NAVY})
ax.set(xlabel="Illustrative value", title="A box plot is a picture of numerical position and spread",
       xlim=(0, 16))
ax.set_yticks([1], ["Example"])
plt.show()

### Quartiles of the real penguins

Unlike the previous example, body mass has a unit: **grams**. We can calculate a five-number summary and IQR for each species, then compare what the box represents for each group.

In [ ]:
quartile_table = (penguins.groupby("species")["body_mass_g"]
                  .agg(n="count", min_g="min", q1_g=lambda s: s.quantile(.25),
                       median_g="median", q3_g=lambda s: s.quantile(.75), max_g="max")
                  .reindex(SPECIES_ORDER))
quartile_table["IQR_g"] = quartile_table["q3_g"] - quartile_table["q1_g"]
display(quartile_table.round(1))

**Interpretation:** For example, Gentoo has $Q_1=4,700$ g and $Q_3=5,500$ g, so its middle-half spread is **800 g**. Chinstrap has a smaller IQR of **462.5 g**. This is a statement about the *middle 50%*, not about the most extreme measurements.

A good statistical description often pairs **median + IQR** when it should be less influenced by extreme values, or **mean + standard deviation** when it should summarize all values around the arithmetic center.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
    <div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">05 · UNUSUAL VALUES</div>
    <div style="font-size:23px;font-weight:700;line-height:1.4;">Unusual observations and the box plot</div>
    <div style="font-size:14px;color:#DDEAF3;margin-top:4px;">When is a value far enough from the middle to deserve attention?</div>
    </div>

### The 1.5 × IQR rule: a useful convention, not a verdict

A conventional box plot often marks values outside the following **fences** as possible outliers:

$$
\text{Lower fence}=Q_1-1.5\times\mathrm{IQR}
$$
$$
\text{Upper fence}=Q_3+1.5\times\mathrm{IQR}.
$$

These numbers are a **rule of thumb** for noticing unusual observations. They are not a test that proves a value is wrong, and the rule can flag more or fewer points depending on the shape of the distribution.

For a small illustrative set **2, 4, 4, 6, 8, 10, 10, 12, 30**:

- $Q_1=4$, $Q_3=10$, so IQR $=6$.
- Lower fence $=4-1.5(6)=-5$.
- Upper fence $=10+1.5(6)=19$.
- The value **30** lies above 19 and is therefore flagged as a *possible outlier*.

**Important box-plot detail:** The whiskers of a conventional Matplotlib box plot stop at the most extreme **actual observed points still inside the fences**. They do **not** automatically extend all the way to −5 or 19. Here the left whisker reaches 2 and the right whisker reaches 12; 30 is shown separately.

In [ ]:
outlier_example = pd.Series([2, 4, 4, 6, 8, 10, 10, 12, 30])
a_q1, a_q3 = outlier_example.quantile([0.25, 0.75])
a_iqr = a_q3 - a_q1
lower_fence = a_q1 - 1.5 * a_iqr
upper_fence = a_q3 + 1.5 * a_iqr
flagged = outlier_example[(outlier_example < lower_fence) | (outlier_example > upper_fence)]
within_fences = outlier_example[outlier_example.between(lower_fence, upper_fence)]
print(f"Fences: [{lower_fence:g}, {upper_fence:g}]")
print("Flagged values:", flagged.tolist())
print("Actual whisker endpoints:", within_fences.min(), "and", within_fences.max())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.1))
ax.boxplot(outlier_example, orientation="horizontal", widths=0.32, patch_artist=True,
           boxprops={"facecolor": "#B2E2DA", "edgecolor": NAVY},
           medianprops={"color": "#B85232", "linewidth": 2.5},
           flierprops={"marker": "o", "markerfacecolor": "#D47040",
                       "markeredgecolor": "#D47040", "markersize": 8})
ax.axvline(lower_fence, color="#A76F3B", linestyle="--", label="Lower fence = -5")
ax.axvline(upper_fence, color="#A76F3B", linestyle="--", label="Upper fence = 19")
ax.set(xlabel="Illustrative value", xlim=(-8, 33),
       title="The upper whisker ends at 12, not at the upper fence (19)")
ax.set_yticks([1], ["Example"])
ax.legend(loc="upper left", fontsize=9)
plt.show()

<div style="border-left:5px solid #18B7AC;background:#EEF8F7;padding:12px 16px;border-radius:5px;"><strong style="color:#102A43;">ACTIVITY 3 — Read the box plot precisely</strong></div>

For the **illustrative values** `2, 4, 4, 6, 8, 10, 10, 12, 30`:

1. Find $Q_1$, the median, $Q_3$, and IQR using Pandas.
2. Compute the two 1.5 × IQR fences and identify any flagged values.
3. Where do the **actual whiskers** end? Why does the right whisker not extend to the upper fence?

*Try the question before opening the suggested answer. The next code cell is yours to edit.*

In [ ]:
# Activity 3: The data are defined for you; complete or edit the calculation.
activity_values = pd.Series([2, 4, 4, 6, 8, 10, 10, 12, 30])
activity_q1 = activity_values.quantile(0.25)
activity_q3 = activity_values.quantile(0.75)
activity_iqr = activity_q3 - activity_q1
print("Q1, median, Q3:", activity_q1, activity_values.median(), activity_q3)
print("IQR:", activity_iqr)
# Now calculate the fences and explain the whisker endpoints.

<details><summary><strong>Suggested answer — click to expand after trying</strong></summary>

Using Pandas' default quantile rule: **Q1 = 4, median = 8, Q3 = 10**, so **IQR = 6**. The fences are **−5** and **19**. Only **30** falls outside them. Among the observed values inside the fences, the minimum is **2** and the maximum is **12**. Thus the whiskers end at **2 and 12**. The fence 19 is a *threshold*, not an observed endpoint.

You can check with:

```python
low = activity_q1 - 1.5 * activity_iqr
high = activity_q3 + 1.5 * activity_iqr
print(low, high)  # -5.0, 19.0
print(activity_values[(activity_values < low) | (activity_values > high)])
```

</details>

### Is the same penguin unusual in every comparison?

Consider body mass across all penguins and then *within each species*. If the three species have different typical masses, an observation may be unusual **within one species** but not unusual in the overall mixture.

We calculate fences **within each species independently**. This is exploratory identification only: we are not removing or correcting any values.

In [ ]:
outlier_rows = []
for species, group in penguins.groupby("species"):
    masses = group["body_mass_g"].dropna()
    q1_s, q3_s = masses.quantile([0.25, 0.75])
    iqr_s = q3_s - q1_s
    low_s, high_s = q1_s - 1.5 * iqr_s, q3_s + 1.5 * iqr_s
    unusual = masses[(masses < low_s) | (masses > high_s)]
    outlier_rows.append({
        "species": species, "valid_n": len(masses),
        "lower_fence_g": low_s, "upper_fence_g": high_s,
        "flagged_n": len(unusual), "flagged_masses_g": unusual.tolist()
    })
outlier_summary = pd.DataFrame(outlier_rows).set_index("species").reindex(SPECIES_ORDER)
display(outlier_summary.round(1))

In [ ]:
all_q1, all_q3 = body_mass.quantile([.25, .75])
all_iqr = all_q3 - all_q1
all_low, all_high = all_q1 - 1.5 * all_iqr, all_q3 + 1.5 * all_iqr
all_unusual = body_mass[(body_mass < all_low) | (body_mass > all_high)]
print(f"Overall fences: {all_low:.0f} to {all_high:.0f} g")
print(f"Observations flagged in the full distribution: {len(all_unusual)}")
print("Within-species counts (shown above):", outlier_summary["flagged_n"].to_dict())

The standard 1.5 × IQR rule flags **no body masses** in the overall mixture, but it flags **two Chinstrap measurements (2,700 g and 4,800 g)** when applied within the Chinstrap group. That is not a contradiction: the reference distribution changed.

**Do not delete these records simply because they are flagged.** Unusual observations may be valid and informative. Confirming data quality and deciding what to do about it belong in **L07**.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
    <div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">06 · TWO ATTRIBUTES</div>
    <div style="font-size:23px;font-weight:700;line-height:1.4;">Measuring relationships</div>
    <div style="font-size:14px;color:#DDEAF3;margin-top:4px;">Do penguins with longer flippers tend to have greater body mass?</div>
    </div>

### From a scatter plot to one numerical measure

In L04, we used a **scatter plot**: each dot was one penguin, positioned by flipper length (horizontal) and body mass (vertical).

If dots tend to rise from left to right, the association is **positive**. If they tend to fall, it is **negative**. But a plot alone does not give a compact number for the **strength** of a linear association.

**Pearson correlation**, written $r$ for a sample, is one such number. It measures how two numerical variables move together in a *linear* pattern. It is calculated from **paired observations** (a row must have both measurements).

### Reading the correlation coefficient

Pearson's $r$ is always between $-1$ and $+1$:

- **$r$ near $+1$:** strong positive *linear* relationship.
- **$r$ near $-1$:** strong negative *linear* relationship.
- **$r$ near $0$:** little or no *linear* relationship.

The closer $|r|$ is to 1, the stronger the linear pattern, but fixed labels such as “weak” and “strong” depend on the field and the question. Correlation has **no physical units** (unlike body mass or standard deviation).

**Two key limits:**

1. Correlation near zero does **not** prove there is no relationship; a strong *curved* relationship may have low Pearson correlation.
2. Correlation does **not** establish causation. A positive value does not mean longer flippers *cause* a penguin to gain mass.

A few unusual points can also affect Pearson correlation strongly. Always consider the scatter plot alongside $r$.

### How the calculation works (intuition first)

To compare two attributes with different units, we first look at their deviations from their means. Pearson correlation is a **normalized measure of how these paired deviations agree**:

$$
r = \frac{\sum_{i=1}^{n}(x_i-\bar{x})(y_i-\bar{y})}
{\sqrt{\sum_{i=1}^{n}(x_i-\bar{x})^2}\,\sqrt{\sum_{i=1}^{n}(y_i-\bar{y})^2}}.
$$

- $x_i$ is one flipper length and $y_i$ is the body mass from the **same penguin**.
- $\bar{x}$ and $\bar{y}$ are the respective means of the paired observations.
- $n$ counts rows with **both** values available.
- In the numerator, pairs with both measurements above their averages contribute **positive products**; pairs with one above and one below contribute **negative products**.
- The denominator rescales these products so the result lies between $-1$ and $1$.

We do not need to calculate this long expression by hand for hundreds of penguins. Understanding the idea is more valuable; Pandas implements it with `.corr()`.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for species in SPECIES_ORDER:
    group = penguins.loc[penguins["species"] == species]
    ax.scatter(group["flipper_length_mm"], group["body_mass_g"],
               alpha=0.74, s=36, label=species,
               color=SPECIES_COLORS[species], edgecolors="none")
ax.set(title=f"Flipper length and body mass (all species): Pearson r = {r_all:.3f}",
       xlabel="Flipper length (mm)", ylabel="Body mass (g)")
ax.legend(title="Species")
plt.show()

In [ ]:
pairs = penguins[["flipper_length_mm", "body_mass_g"]].dropna()
r_all = pairs["flipper_length_mm"].corr(pairs["body_mass_g"])
print("Complete (flipper length, body mass) pairs:", len(pairs))
print(f"Pearson correlation across all species: r = {r_all:.3f}")
display(pairs.corr().round(3))

A Pearson correlation of about **+0.871** indicates a strong positive **linear association in the pooled data**: penguins with longer flippers tend to have greater body mass **in these observations**. This does *not* tell us whether the association has the same strength inside each species.

Also notice that the species have different typical body masses and flipper lengths. When groups differ in both variables, the pooled relationship can look stronger than each within-group relationship. We should examine both.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.7), sharey=True, sharex=True)
for ax, species in zip(axes, SPECIES_ORDER):
    group = penguins.loc[penguins["species"] == species]
    r_s = correlations.loc[species, "Pearson r"]
    ax.scatter(group["flipper_length_mm"], group["body_mass_g"],
               alpha=0.8, s=33, color=SPECIES_COLORS[species])
    ax.set(title=f"{species}: r = {r_s:.3f}", xlabel="Flipper length (mm)")
axes[0].set_ylabel("Body mass (g)")
fig.suptitle("Within-species relationships: compare them with the pooled plot")
plt.tight_layout()
plt.show()

In [ ]:
within_species_r = []
for species in SPECIES_ORDER:
    g = penguins.loc[penguins["species"] == species,
                     ["flipper_length_mm", "body_mass_g"]].dropna()
    within_species_r.append({"Species": species, "Pairs (n)": len(g),
                             "Pearson r": g["flipper_length_mm"].corr(g["body_mass_g"])})
correlations = pd.DataFrame(within_species_r).set_index("Species")
display(correlations.round(3))

In this dataset the approximate within-species correlations are **+0.468 (Adelie)**, **+0.642 (Chinstrap)**, and **+0.703 (Gentoo)**, all smaller than the pooled **+0.871**. These numbers describe different questions: *across all measured penguins* versus *among penguins of the same species*.

We cannot conclude that changing a penguin's flipper length would change its mass. Nor should we label an association “unimportant” simply because one subgroup has a smaller correlation.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
    <div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">07 · USING STATISTICS</div>
    <div style="font-size:23px;font-weight:700;line-height:1.4;">Choosing useful summaries</div>
    <div style="font-size:14px;color:#DDEAF3;margin-top:4px;">Which few statistics help us answer the question without hiding the story?</div>
    </div>

### Choose the measure before you choose the Python method

A long table of statistics is not automatically a useful analysis. Begin with the **question**:

| If your question is... | Consider reporting... | Check as well... |
|:--|:--|:--|
| What is the arithmetic center? | Mean | Histogram; unusual observations |
| What is the middle observed position? | Median | Distribution shape and count |
| What category is most common? | Mode / frequency counts | Other category frequencies |
| How spread are values around the mean? | Standard deviation (with convention stated) | Mean, units, unusual values |
| How wide is the middle half? | Q1, Q3, IQR | Median and box plot |
| What are the observed extremes? | Minimum, maximum / range | Whether extremes are unusual |
| Do two numerical attributes vary together linearly? | Pearson $r$ | Scatter plot; groups; pair count |

**Mean + standard deviation** and **median + IQR** are complementary, not competing, descriptions. One may be more helpful depending on how the data are distributed and what the audience needs to know.

### Revisiting `.describe()` — now we can explain every row

In L03, we used `.describe()` before learning what its statistics meant. We now know how to interpret this compact Pandas summary. Prof. Liotta also uses `.describe()` when exploring air-quality and graduate-admissions measurements.

Look at **one numerical attribute** rather than a long table of unrelated columns. Predict which of today's concepts each row represents before running the cell.


In [ ]:
# We met this Pandas method in L03. Can you explain every line now?
penguins["body_mass_g"].describe().round(1)


Read this table as follows: `count` is the number of **available measurements**, not the number of all rows; `mean` describes arithmetic center; `std` is the **sample standard deviation**; `min` and `max` are the observed extremes; `25%`, `50%`, and `75%` are $Q_1$, the **median**, and $Q_3$.

Notice what this command does **not** report directly: the mode, range, variance, or IQR. Nor does it explain subgroup differences. We still need to **choose** measures and interpret them according to our question.


### A compact, evidence-based comparison

Suppose someone asks: “How does body mass differ among Adelie, Chinstrap, and Gentoo penguins?”

A good summary might report **count, mean, median, standard deviation, and IQR** for each species. This answers both center and spread, and it avoids confusing an overall average with a species-specific value.

In [ ]:
summary = (penguins.groupby("species")["body_mass_g"]
           .agg(n="count", mean_g="mean", median_g="median", std_g="std",
                q1_g=lambda s: s.quantile(.25), q3_g=lambda s: s.quantile(.75))
           .reindex(SPECIES_ORDER))
summary["IQR_g"] = summary["q3_g"] - summary["q1_g"]
display(summary[["n", "mean_g", "median_g", "std_g", "IQR_g"]].round(1))

**Example of a careful written conclusion:**

> Among penguins with recorded body masses, Gentoo has a higher mean body mass (about 5,076 g) than Adelie (3,701 g) or Chinstrap (3,733 g). Gentoo also has a larger middle-half spread (IQR 800 g) than the other two species. The sample sizes are 123, 151, and 68, respectively. 

Notice what we did **not** say: that the higher Gentoo mean proves why Gentoo is heavier; that every Gentoo outweighs every Adelie; or that two similar means imply two similar distributions.

### A final check: what remains outside these numbers?

A summary can hide **subgroups, unusual values, and non-linear patterns**. A correlation alone says nothing about causation. A mean alone says nothing about spread. A standard deviation alone says nothing about the shape of a histogram. Missing observations can also affect which records contributed to a result.

That is why descriptive statistics should work **with** visual exploration and a clear question, not replace them. We are building habits for later machine learning: understanding our inputs, checking distributions, and explaining what the available evidence really supports.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
    <div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">08 · BRING IT TOGETHER</div>
    <div style="font-size:23px;font-weight:700;line-height:1.4;">Recap and next steps</div>
    <div style="font-size:14px;color:#DDEAF3;margin-top:4px;">We can describe observed data. How do we reason when our observations are only a sample?</div>
    </div>

### Where we go next — L06: Probability, Distributions, and Sampling

Today we **described the observations we have**. In L06, we will ask a different question:

> **What happens when we observe only a sample of a larger population, and how can we reason about uncertainty?**

That will introduce probability, distributions, and sampling. For now, remember that even precise descriptive statistics cannot automatically guarantee conclusions beyond the data we measured.

### Sources, acknowledgements, and further reading

- **Palmer Penguins dataset:** Horst, A. M., Hill, A. P., & Gorman, K. B. (2020), *palmerpenguins: Palmer Archipelago (Antarctica) penguin data* ([dataset and documentation](https://allisonhorst.github.io/palmerpenguins/), [DOI: 10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218)). Original ecological measurements: Gorman, K. B., Williams, T. D., & Fraser, W. R. (2014), *PLoS ONE*. The bundled `penguins.csv` is the standard 344-row, 8-column version, supplied locally for offline teaching; the dataset is available under CC0.
- **Teaching inspirations:** Selected concepts were adapted from the teaching archive of **Prof. Antonio Liotta**.
- **Pandas documentation:** [`Series.mean`](https://pandas.pydata.org/docs/reference/api/pandas.Series.mean.html), [`Series.std`](https://pandas.pydata.org/docs/reference/api/pandas.Series.std.html), [`Series.quantile`](https://pandas.pydata.org/docs/reference/api/pandas.Series.quantile.html), [`Series.corr`](https://pandas.pydata.org/docs/reference/api/pandas.Series.corr.html).
- **NumPy documentation:** [`numpy.std`](https://numpy.org/doc/stable/reference/generated/numpy.std.html) (note the population default, `ddof=0`).
- **Matplotlib documentation:** [`boxplot`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.boxplot.html) (whiskers, fences, fliers).
